In [4]:
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent
RESULTS_DIR = ROOT_DIR / "results"


In [10]:
import pandas as pd

FOLDER = "Cut_Crawli1_BB006_0_blur"

df = pd.read_csv(RESULTS_DIR / FOLDER / f"baby_{FOLDER}.csv")
df.head()

,frame,person_id,keypoint,x,y,confidence
0,92,2,Nose,1404.869629,785.155701,0.801727
1,92,2,Left Eye,1404.670898,766.540771,0.947913
2,92,2,Right Eye,1393.224365,770.255981,0.092514
3,92,2,Left Ear,1454.345337,708.823181,0.990103
4,92,2,Right Ear,1409.976929,717.718567,0.018369


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


# ============================================================
# PARAMÈTRES
# ============================================================

CONF_THRESHOLD = 0.5

# Nombre maximum de frames consécutives pouvant être interpolées
MAX_INTERPOLATION_GAP = 10

# Paramètres du filtre Savitzky-Golay
SMOOTHING_WINDOW = 11
SMOOTHING_POLYORDER = 2

# Distance maximale autorisée entre deux frames.
# Elle sera multipliée par la taille du corps.
MAX_MOVEMENT_RATIO = 0.25


# ============================================================
# OUTILS
# ============================================================

def calculate_body_scale(group):
    """
    Estime la taille du bébé dans l'image à partir des hanches
    et des épaules.
    """

    points = {
        row["keypoint"]: (row["x"], row["y"])
        for _, row in group.iterrows()
    }

    required = [
        "Left Shoulder",
        "Right Shoulder",
        "Left Hip",
        "Right Hip",
    ]

    if not all(k in points for k in required):
        return np.nan

    left_shoulder = np.array(points["Left Shoulder"])
    right_shoulder = np.array(points["Right Shoulder"])

    left_hip = np.array(points["Left Hip"])
    right_hip = np.array(points["Right Hip"])

    shoulder_width = np.linalg.norm(
        left_shoulder - right_shoulder
    )

    hip_width = np.linalg.norm(
        left_hip - right_hip
    )

    scale = np.nanmedian([
        shoulder_width,
        hip_width
    ])

    return scale


def clean_keypoint_trajectory(
    trajectory,
    confidence,
    max_interpolation_gap=10
):
    """
    Nettoie une trajectoire 1D :
    1. supprime les points de faible confiance
    2. interpole les petits trous
    3. lisse les données valides
    """

    values = np.asarray(
        trajectory,
        dtype=float
    ).copy()

    # --------------------------------------------------------
    # 1. Faible confiance
    # --------------------------------------------------------

    values[confidence < CONF_THRESHOLD] = np.nan

    # --------------------------------------------------------
    # 2. Interpolation
    # --------------------------------------------------------

    series = pd.Series(values)

    series = series.interpolate(
        method="linear",
        limit=max_interpolation_gap,
        limit_area="inside"
    )

    values = series.to_numpy()

    # --------------------------------------------------------
    # 3. Vérification
    # --------------------------------------------------------

    valid = ~np.isnan(values)

    # Pas assez de données pour filtrer
    if valid.sum() < 5:
        return values

    # --------------------------------------------------------
    # 4. Lissage
    # --------------------------------------------------------

    # On ne peut pas donner de NaN à savgol_filter.
    # On ne filtre donc que les parties valides.

    first = np.where(valid)[0][0]
    last = np.where(valid)[0][-1]

    segment = values[first:last + 1]

    # S'il reste des NaN à l'intérieur, on les interpole
    segment = (
        pd.Series(segment)
        .interpolate(method="linear")
        .ffill()
        .bfill()
        .to_numpy()
    )

    # Nombre de points disponibles
    n = len(segment)

    if n < SMOOTHING_POLYORDER + 2:
        values[first:last + 1] = segment
        return values

    # Fenêtre impaire
    window = min(
        SMOOTHING_WINDOW,
        n if n % 2 == 1 else n - 1
    )

    # La fenêtre doit être > polyorder
    if window <= SMOOTHING_POLYORDER:
        values[first:last + 1] = segment
        return values

    # --------------------------------------------------------
    # 5. Savitzky-Golay
    # --------------------------------------------------------

    smoothed = savgol_filter(
        segment,
        window_length=window,
        polyorder=SMOOTHING_POLYORDER
    )

    values[first:last + 1] = smoothed

    return values


# ============================================================
# NETTOYAGE
# ============================================================

def clean_dataframe(df):

    df = df.copy()

    # Colonnes supplémentaires
    df["is_outlier"] = False
    df["x_clean"] = df["x"]
    df["y_clean"] = df["y"]

    # --------------------------------------------------------
    # 1. Faible confiance
    # --------------------------------------------------------

    low_confidence = (
        df["confidence"] < CONF_THRESHOLD
    )

    df.loc[
        low_confidence,
        "is_outlier"
    ] = True

    df.loc[
        low_confidence,
        ["x_clean", "y_clean"]
    ] = np.nan

    # --------------------------------------------------------
    # 2. Traitement personne par personne / keypoint
    # --------------------------------------------------------

    for (person_id, keypoint), group in df.groupby(
        ["person_id", "keypoint"],
        sort=False
    ):

        group = group.sort_values("frame")

        indices = group.index.to_numpy()

        x = group["x_clean"].to_numpy()
        y = group["y_clean"].to_numpy()

        confidence = group["confidence"].to_numpy()

        # ----------------------------------------------------
        # Détection des gros sauts
        # ----------------------------------------------------

        for i in range(1, len(group)):

            if np.isnan(x[i]) or np.isnan(y[i]):
                continue

            if np.isnan(x[i - 1]) or np.isnan(y[i - 1]):
                continue

            dx = x[i] - x[i - 1]
            dy = y[i] - y[i - 1]

            distance = np.sqrt(
                dx ** 2 + dy ** 2
            )

            # Estimation grossière de l'échelle du corps
            current_frame = df[
                df["frame"] == group.iloc[i]["frame"]
            ]

            body_scale = calculate_body_scale(
                current_frame
            )

            if np.isnan(body_scale):
                continue

            max_distance = (
                MAX_MOVEMENT_RATIO *
                body_scale
            )

            if distance > max_distance:

                idx = indices[i]

                df.loc[idx, "is_outlier"] = True

                df.loc[
                    idx,
                    ["x_clean", "y_clean"]
                ] = np.nan

        # ----------------------------------------------------
        # Interpolation + lissage
        # ----------------------------------------------------

        cleaned_x = clean_keypoint_trajectory(
            df.loc[indices, "x_clean"].to_numpy(),
            confidence
        )

        cleaned_y = clean_keypoint_trajectory(
            df.loc[indices, "y_clean"].to_numpy(),
            confidence
        )

        df.loc[indices, "x_clean"] = cleaned_x
        df.loc[indices, "y_clean"] = cleaned_y

    return df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    RESULTS_DIR = Path("../results")

    # À modifier selon ton dossier
    folder = FOLDER

    input_file = (
        RESULTS_DIR /
        folder /
        f"baby_{folder}.csv"
    )

    output_file = (
        RESULTS_DIR /
        folder /
        f"baby_{folder}_clean.csv"
    )

    print(f"Reading: {input_file}")

    df = pd.read_csv(input_file)

    print(
        f"{len(df)} keypoints loaded"
    )

    clean_df = clean_dataframe(df)

    clean_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Clean data saved to: {output_file}"
    )

    print(
        f"Outliers detected: "
        f"{clean_df['is_outlier'].sum()}"
    )

Reading: ..\results\Cut_Crawli1_BB006_0_blur\baby_Cut_Crawli1_BB006_0_blur.csv
65518 keypoints loaded
Clean data saved to: ..\results\Cut_Crawli1_BB006_0_blur\baby_Cut_Crawli1_BB006_0_blur_clean.csv
Outliers detected: 22683


In [11]:
# visualize_clean.py

from pathlib import Path

import cv2
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_DIR = ROOT_DIR / "results"
VIDEOS_DIR = ROOT_DIR / "videos"

# Nom du fichier vidéo
VIDEO_EXTENSION = ".mp4"

# Afficher uniquement les points suffisamment fiables
CONF_THRESHOLD = 0.5

# Rayon des points
POINT_RADIUS = 5

# Épaisseur des lignes du squelette
LINE_THICKNESS = 2


# ============================================================
# SQUELETTE COCO
# ============================================================

SKELETON = [
    ("Nose", "Left Eye"),
    ("Nose", "Right Eye"),
    ("Left Eye", "Left Ear"),
    ("Right Eye", "Right Ear"),

    ("Left Shoulder", "Right Shoulder"),

    ("Left Shoulder", "Left Elbow"),
    ("Left Elbow", "Left Wrist"),

    ("Right Shoulder", "Right Elbow"),
    ("Right Elbow", "Right Wrist"),

    ("Left Shoulder", "Left Hip"),
    ("Right Shoulder", "Right Hip"),

    ("Left Hip", "Right Hip"),

    ("Left Hip", "Left Knee"),
    ("Left Knee", "Left Ankle"),

    ("Right Hip", "Right Knee"),
    ("Right Knee", "Right Ankle"),
]


# ============================================================
# OUTILS
# ============================================================

def draw_keypoints(frame, person_df):
    """
    Dessine les keypoints nettoyés et le squelette.
    """

    points = {}

    for _, row in person_df.iterrows():

        x = row["x_clean"]
        y = row["y_clean"]

        if pd.isna(x) or pd.isna(y):
            continue

        if row["confidence"] < CONF_THRESHOLD:
            continue

        keypoint = row["keypoint"]

        points[keypoint] = (
            int(round(x)),
            int(round(y))
        )

    # --------------------------------------------------------
    # Squelette
    # --------------------------------------------------------

    for kp1, kp2 in SKELETON:

        if kp1 not in points or kp2 not in points:
            continue

        p1 = points[kp1]
        p2 = points[kp2]

        cv2.line(
            frame,
            p1,
            p2,
            (0, 255, 0),
            LINE_THICKNESS
        )

    # --------------------------------------------------------
    # Keypoints
    # --------------------------------------------------------

    for keypoint, (x, y) in points.items():

        cv2.circle(
            frame,
            (x, y),
            POINT_RADIUS,
            (0, 0, 255),
            -1
        )

    return frame


# ============================================================
# VISUALISATION
# ============================================================

def visualize():

    folder = RESULTS_DIR / FOLDER

    csv_path = (
        folder /
        f"baby_{FOLDER}_clean.csv"
    )

    video_path = (
        VIDEOS_DIR /
        f"{FOLDER}{VIDEO_EXTENSION}"
    )

    output_path = (
        folder /
        f"visualization_{FOLDER}.avi"
    )

    # --------------------------------------------------------
    # Vérifications
    # --------------------------------------------------------

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found:\n{csv_path}"
        )

    if not video_path.exists():
        raise FileNotFoundError(
            f"Video not found:\n{video_path}"
        )

    print(f"CSV    : {csv_path}")
    print(f"Video  : {video_path}")
    print(f"Output : {output_path}")

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    df = pd.read_csv(csv_path)

    # --------------------------------------------------------
    # Vidéo
    # --------------------------------------------------------

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Unable to open video:\n{video_path}"
        )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    # --------------------------------------------------------
    # VideoWriter
    # --------------------------------------------------------

    fourcc = cv2.VideoWriter_fourcc(
        *"XVID"
    )

    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (width, height)
    )

    # --------------------------------------------------------
    # Traitement frame par frame
    # --------------------------------------------------------

    frame_number = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # ----------------------------------------------------
        # Keypoints de cette frame
        # ----------------------------------------------------

        frame_df = df[
            df["frame"] == frame_number
        ]

        # ----------------------------------------------------
        # Plusieurs personnes possibles
        # ----------------------------------------------------

        for person_id, person_df in frame_df.groupby(
            "person_id"
        ):

            frame = draw_keypoints(
                frame,
                person_df
            )

            # Affichage de l'ID
            valid_points = person_df[
                person_df["x_clean"].notna()
                & person_df["y_clean"].notna()
            ]

            if not valid_points.empty:

                x = int(
                    valid_points["x_clean"].mean()
                )

                y = int(
                    valid_points["y_clean"].mean()
                )

                cv2.putText(
                    frame,
                    f"ID: {int(person_id)}",
                    (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 0),
                    2
                )

        # ----------------------------------------------------
        # Texte frame
        # ----------------------------------------------------

        cv2.putText(
            frame,
            f"Frame: {frame_number}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

        writer.write(frame)

        # Affichage temps réel
        cv2.imshow(
            "Clean keypoints",
            frame
        )

        key = cv2.waitKey(1)

        if key == 27:  # ESC
            break

        frame_number += 1

        if frame_number % 100 == 0:
            print(
                f"{frame_number}/{total_frames}"
            )

    # --------------------------------------------------------
    # Nettoyage
    # --------------------------------------------------------

    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print()
    print(
        f"Visualization saved to:\n{output_path}"
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    visualize()

CSV    : C:\Users\billy\Documents\Stage_2026\BabyMove\results\Cut_Crawli1_BB006_0_blur\baby_Cut_Crawli1_BB006_0_blur_clean.csv
Video  : C:\Users\billy\Documents\Stage_2026\BabyMove\videos\Cut_Crawli1_BB006_0_blur.mp4
Output : C:\Users\billy\Documents\Stage_2026\BabyMove\results\Cut_Crawli1_BB006_0_blur\visualization_Cut_Crawli1_BB006_0_blur.avi
100/3946
200/3946
300/3946
400/3946
500/3946
600/3946
700/3946
800/3946
900/3946
1000/3946
1100/3946
1200/3946
1300/3946
1400/3946
1500/3946
1600/3946
1700/3946
1800/3946
1900/3946
2000/3946
2100/3946
2200/3946
2300/3946
2400/3946
2500/3946
2600/3946
2700/3946
2800/3946
2900/3946
3000/3946
3100/3946
3200/3946
3300/3946
3400/3946
3500/3946
3600/3946
3700/3946
3800/3946
3900/3946

Visualization saved to:
C:\Users\billy\Documents\Stage_2026\BabyMove\results\Cut_Crawli1_BB006_0_blur\visualization_Cut_Crawli1_BB006_0_blur.avi
